# Options OHLCV Analysis Sample

Query NIFTY option candles from ClickHouse and plot close prices by strike and option type.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from app.config.settings import get_settings
from app.db.client import create_clickhouse_client

settings = get_settings()
client = create_clickhouse_client(settings)


In [ ]:
underlying = "NIFTY"
expiry_date = "2026-05-07"

query = """
SELECT
    datetime,
    strike_price,
    toString(option_type) AS option_type,
    close,
    volume,
    open_interest
FROM options_ohlcv
WHERE underlying = {underlying:String}
  AND expiry_date = {expiry_date:Date}
ORDER BY datetime, strike_price, option_type
"""

result = client.query(query, parameters={"underlying": underlying, "expiry_date": expiry_date})
df = pd.DataFrame(result.result_rows, columns=result.column_names)
df.head()


In [ ]:
if not df.empty:
    pivot = df.pivot_table(index="datetime", columns=["strike_price", "option_type"], values="close")
    ax = pivot.plot(figsize=(14, 6), title=f"{underlying} options close by strike")
    ax.set_ylabel("Close")
    plt.show()
